![Cabecera](../../../Sprint_11/assets/cabecera_thebridge.png)

# Workout — HITL: aprobar plan

## Objetivo

**HITL** = *Human-in-the-loop*: una persona **aprueba o rechaza** un paso sensible antes de que el sistema lo ejecute.

Aquí el grafo propone un plan, **se pausa**, tú decides, y luego reanuda.

```text
START → proponer → (PAUSA) → aplicar → END
```

Patrón de este notebook: `interrupt_before` + checkpointer (obligatorio: sin él no hay dónde guardar la pausa).

## Índice

1. Setup
2. Grafo proponer → (pausa) → aplicar
3. Primer `invoke` — pausa antes de aplicar
4. Humano aprueba y reanudamos
5. Variante: rechazar
6. Para llevarnos

Teoría: [`03_human_in_the_loop.md`](../../01_Teoria/02_HITL_guardrails_y_persistencia/03_human_in_the_loop.md)

Prerrequisito: memoria y checkpoints (notebook anterior).

**Requisito:** `GEMINI_API_KEY`


## 1. Setup

Misma pila que memoria / ReAct: LangGraph + Gemini.

Descomenta la celda de `%pip` solo la primera vez o si falta algún paquete.

In [ ]:
# %pip install -qU langgraph langchain-google-genai langchain-core python-dotenv

In [1]:
# 1) Intentamos leer GEMINI_API_KEY desde un fichero .env
# 2) Si no está, la pedimos oculta con getpass
import os
from getpass import getpass

from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Pega tu GEMINI_API_KEY: ")

API_KEY = os.environ["GEMINI_API_KEY"]
print("GEMINI_API_KEY:", "sí" if API_KEY else "no")

GEMINI_API_KEY: sí


## 2. Grafo: proponer → (pausa) → aplicar

```text
START → proponer → aplicar → END
                 ↑
         interrupt_before aquí
```

| Pieza | Qué hace |
|-------|----------|
| Nodo `proponer` | El LLM escribe un plan (3 viñetas) |
| `interrupt_before=["aplicar"]` | El grafo **no entra** en `aplicar` hasta que reanudemos |
| Nodo `aplicar` | Si `aprobado` → cierra el plan; si no → mensaje de rechazo |
| `MemorySaver` | Guarda la pausa |

El State no usa `messages` aquí: pedimos un dominio simple (`pedido`, `plan`, `aprobado`, `resultado`) para ver solo el HITL.

In [2]:
from typing import TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=API_KEY,
    temperature=0,
)


# Estado del flujo HITL (campos claros, sin historial de chat)
class EstadoPlan(TypedDict):
    pedido: str      # lo que pide el usuario
    plan: str        # texto que genera el nodo proponer
    aprobado: bool   # decisión humana (False hasta que update_state)
    resultado: str   # salida final del nodo aplicar


def nodo_proponer(estado: EstadoPlan) -> dict:
    """LLM escribe el plan. Deja aprobado=False (aún no hay humano)."""
    prompt = [
        SystemMessage(
            content=(
                "Propones un plan de tarde cultural en 3 viñetas cortas. "
                "Solo el plan, sin saludos."
            )
        ),
        HumanMessage(content=estado["pedido"]),
    ]
    plan = llm.invoke(prompt).content
    if not isinstance(plan, str):
        plan = str(plan)
    print("[proponer] plan listo")
    return {"plan": plan, "aprobado": False, "resultado": ""}


def nodo_aplicar(estado: EstadoPlan) -> dict:
    """Solo se ejecuta DESPUÉS de la pausa HITL (cuando reanudamos)."""
    if not estado.get("aprobado"):
        print("[aplicar] rechazado → no se cierra el plan")
        return {"resultado": "Plan rechazado. No se aplica."}
    print("[aplicar] aprobado → cerramos")
    return {"resultado": f"Plan APROBADO y aplicado:\n{estado['plan']}"}


grafo = StateGraph(EstadoPlan)
grafo.add_node("proponer", nodo_proponer)
grafo.add_node("aplicar", nodo_aplicar)
grafo.add_edge(START, "proponer")
grafo.add_edge("proponer", "aplicar")
grafo.add_edge("aplicar", END)

# Checkpointer OBLIGATORIO con interrupt_before (guarda dónde nos paramos)
app = grafo.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["aplicar"],  # pausa justo antes de este nodo
)
print("Compilado con interrupt_before=['aplicar']")

Compilado con interrupt_before=['aplicar']


## 3. Primer `invoke` — se detiene antes de aplicar

Lanzamos el pedido. El nodo `proponer` se ejecuta; **antes** de `aplicar` el grafo para.

Receta fija del `config` (igual que en memoria):

```python
config = {"configurable": {"thread_id": "hitl-demo-1"}}
```

Tras la pausa miramos `get_state`:

- `snap.next` → debería incluir `aplicar` (lo que falta por ejecutar).
- `plan` → ya relleno.
- `resultado` → aún vacío (no hemos pasado por `aplicar`).

In [3]:
# Mismo thread_id = mismo hilo (necesario para pausar y reanudar después)
config = {"configurable": {"thread_id": "hitl-demo-1"}}

# Este invoke NO termina el grafo: se corta antes de "aplicar"
parcial = app.invoke(
    {
        "pedido": "Tarde en Madrid: museo + cena vegetariana, presupuesto medio.",
        "plan": "",
        "aprobado": False,
        "resultado": "",
    },
    config=config,
)

# ¿Dónde estamos? get_state lee el checkpoint de la pausa
snap = app.get_state(config)
print("next (debe incluir 'aplicar'):", snap.next)
print("\nPLAN PROPUESTO:\n", snap.values.get("plan"))
print("\nresultado aún:", repr(snap.values.get("resultado")))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[proponer] plan listo
next (debe incluir 'aplicar'): ('aplicar',)

PLAN PROPUESTO:
 Aquí tienes el plan:

1.  **Tarde de Arte Moderno:** Visita el Museo Reina Sofía para admirar el Guernica y sus colecciones de arte contemporáneo. Aprovecha la entrada gratuita en las últimas horas de la tarde (si aplica) o el precio reducido.
2.  **Paseo y Aperitivo:** Tras el museo, un breve paseo por el barrio de Lavapiés o Las Letras. Disfruta de una caña o un vino en una terraza con encanto, empapándote del ambiente madrileño.
3.  **Cena Vegetariana:** Dirígete a un restaurante con opciones vegetarianas creativas en Malasaña o La Latina. Busca locales con "menú del día" o platos para compartir que ofrezcan una experiencia culinaria rica y asequible.

resultado aún: ''


## 4. Humano aprueba y reanudamos

Simulamos el botón **Aprobar** de una UI (p. ej. Streamlit):

1. `update_state` — escribimos en el State `aprobado=True` (decisión humana, no del LLM).
2. `invoke(None, config=...)` — **reanuda** el mismo hilo sin un pedido nuevo. `None` = "sigue desde la pausa".

Después, `aplicar` corre y `next` debería quedar vacío (grafo terminado).

In [4]:
# 1) Decisión humana → entra en el State del hilo hitl-demo-1
app.update_state(config, {"aprobado": True})

# 2) Reanudar: None = no hay input nuevo; continúa desde interrupt_before
final = app.invoke(None, config=config)

print("RESULTADO:\n", final.get("resultado"))
# Tupla vacía / sin 'aplicar' ⇒ el grafo ya terminó
print("next tras reanudar:", app.get_state(config).next)

[aplicar] aprobado → cerramos
RESULTADO:
 Plan APROBADO y aplicado:
Aquí tienes el plan:

1.  **Tarde de Arte Moderno:** Visita el Museo Reina Sofía para admirar el Guernica y sus colecciones de arte contemporáneo. Aprovecha la entrada gratuita en las últimas horas de la tarde (si aplica) o el precio reducido.
2.  **Paseo y Aperitivo:** Tras el museo, un breve paseo por el barrio de Lavapiés o Las Letras. Disfruta de una caña o un vino en una terraza con encanto, empapándote del ambiente madrileño.
3.  **Cena Vegetariana:** Dirígete a un restaurante con opciones vegetarianas creativas en Malasaña o La Latina. Busca locales con "menú del día" o platos para compartir que ofrezcan una experiencia culinaria rica y asequible.
next tras reanudar: ()


## 5. Variante: rechazar

Otro `thread_id` (`hitl-demo-2`) = otra conversación.

Misma pausa, pero `aprobado=False`. El nodo `aplicar` igual se ejecuta al reanudar; dentro decide no cerrar el plan y deja un mensaje de rechazo.

In [5]:
# Hilo nuevo (no mezcla con hitl-demo-1)
config2 = {"configurable": {"thread_id": "hitl-demo-2"}}

# 1) Proponer + pausa
app.invoke(
    {
        "pedido": "Solo un concierto esta noche.",
        "plan": "",
        "aprobado": False,
        "resultado": "",
    },
    config=config2,
)
print("Plan pendiente:\n", app.get_state(config2).values.get("plan"))

# 2) Humano rechaza
app.update_state(config2, {"aprobado": False})

# 3) Reanudar → aplicar ve aprobado=False
out = app.invoke(None, config=config2)
print("\nRESULTADO:\n", out.get("resultado"))

[proponer] plan listo
Plan pendiente:
 **Viñeta 1:**
Llegada al auditorio. Luces tenues, el murmullo de la gente buscando sus asientos. El programa en mano, la expectación crece.

**Viñeta 2:**
Las luces se apagan. Silencio. La orquesta afina, luego el primer acorde resuena, llenando la sala con la magia de la música en vivo.

**Viñeta 3:**
El último compás se desvanece. Un estruendo de aplausos. La emoción perdura mientras la gente se levanta, llevando la melodía en el alma.
[aplicar] rechazado → no se cierra el plan

RESULTADO:
 Plan rechazado. No se aplica.


## 6. Para llevarnos

| Pieza | Rol |
|-------|-----|
| `interrupt_before` | Para el grafo **antes** de un nodo sensible |
| Checkpointer (`MemorySaver`) | Guarda la pausa; sin él no hay HITL de este tipo |
| `thread_id` + `config` | Mismo hilo al pausar y al reanudar |
| `update_state` | Inyecta la decisión humana en el State |
| `invoke(None)` | Reanuda desde la pausa (sin pedido nuevo) |
| `get_state` / `snap.next` | Ver plan y qué nodo falta |

Flujo que hemos practicado: **proponer → pausa → (humano) → aplicar**.

Recuerda que con HITL, el grafo se pausa antes de ejecutar un nodo sensible y se reanuda tras una decisión humana. La decisión humana se inyecta en el State con `update_state` y se reanuda con `invoke(None)`. Puede ser una decisión positiva o negativa. En función de eso, el grafo se cierra o no.